# 03 · Does efficiency pressure change *who the model is*?  And is it one knob?
**Topic 2 (character) × Topic 4 (reasoning efficiency).**
*H1.* λ>0 moves non-targeted dispositions (abstention on underdetermined problems, caving to pushback, calibration) beyond λ=0 RL — dose-dependently.
*H2.* RL's residual shift projects mostly onto an "effort" direction that is not orthogonal to humility/agreeableness. Tests: geometry + steering×RL 2×2.

In [ ]:
import sys, json, os, glob, time; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from lab import profiles, local, envs, collective, cost
P = profiles.get(); MAIN = P["main"]; HF = {t: v[0] for t, v in P["models"].items()}; FINAL = P["steps"]
SPEC = f"spec_{P['name']}.json"
print("PROFILE:", P["name"], "| GPU:", P["gpu"], "| cap $", P["cap"], "| main model:", HF[MAIN], "| steps:", FINAL)

In [ ]:
spec = json.load(open(SPEC))
runs = sorted(k for k in json.load(open(local.runs_file())) if k.startswith(MAIN + "_lam")); print(runs)

### Checkpoint batteries (tiers from the profile)

In [ ]:
for r in runs:
    for s in P["ckpts"]:
        t = f"ev_{r}_{s}"
        local.launch("eval_ckpt", dict(name=r, ckpt=s, model=HF[MAIN], tier=P["tier_final"] if s == FINAL else P["tier_ckpt"], spec=spec, out_tag=t), key=t)
print(local.wait(f"ev_{MAIN}", every=90)); local.pull("/eval")

In [ ]:
rows = []
for f in local.find(f"ev_{MAIN}_*.json"):
    j = json.load(open(f)); n = os.path.basename(f)[3:-5]
    if n == f"{MAIN}_base": run, step, lam, seed = "base", 0, 0.0, -1
    else:
        run, step = n.rsplit("_", 1); step = int(step); lam = float(run.split("lam")[1].split("_")[0]); seed = int(run.split("_s")[-1])
    rows.append(dict(run=run, lam=lam, seed=seed, step=step, **j["metrics"]))
E = pd.DataFrame(rows).sort_values(["lam", "seed", "step"]); E.to_csv(f"evals_{P['name']}.csv", index=False)
fin = E[(E.step == FINAL) | (E.run == "base")]
cols = [c for c in ["acc_id", "acc_ood", "len_id", "abstain_recall", "abstain_false", "fabricate", "flip_user", "flip_peer", "changed_neutral", "calib_auroc", "calib_ece"] if c in fin]
print(fin.groupby(["lam", "seed"])[cols].mean().round(3))

In [ ]:
def boot(recs, key, filt=lambda r: True, B=2000, seed=0):
    v = np.array([float(r[key]) for r in recs if filt(r)]); rng = np.random.default_rng(seed)
    m = [rng.choice(v, len(v)).mean() for _ in range(B)]; return v.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)
panels = [("abst", "answered_und", lambda r: r["und"], "abstain recall (UND items)"),
          ("push", "flip", lambda r: r["kind"] == "user", "cave to user pushback"),
          ("push", "flip", lambda r: r["kind"] == "peer", "cave to peer answer"),
          ("acc", "correct", lambda r: r["group"] == "id", "accuracy (in-dist)")]
def final_recs(lam):
    recs = []
    for f in local.find(f"ev_{MAIN}_lam{lam}_s*_{FINAL}.json"): recs += json.load(open(f))["records"]
    return recs
base_recs = json.load(open(local.find(f"ev_{MAIN}_base.json")[0]))["records"]
fig, ax = plt.subplots(1, 4, figsize=(15, 3.2))
for a, (probe, key, filt, title) in zip(ax, panels):
    m, lo, hi = boot([r for r in base_recs if r["probe"] == probe], key, filt); a.axhline(m, ls="--", color="gray"); a.axhspan(lo, hi, color="gray", alpha=.15)
    for lam in sorted(fin[fin.run != "base"].lam.unique()):
        recs = [r for r in final_recs(lam) if r["probe"] == probe]
        if recs: m, lo, hi = boot(recs, key, filt); a.errorbar(lam, m, yerr=[[m - lo], [hi - m]], fmt="o", capsize=3, color="C0")
    a.set_title(title + "  (dashed = base)", fontsize=8); a.set_xlabel("λ")
plt.tight_layout(); plt.show()

### Geometry: where does RL move the residual stream, in units of persona contrast?

In [ ]:
runs_geo = [dict(tag=f"{r}_{s}", name=r, ckpt=s) for r in runs for s in (P["early"], FINAL)]
geo = local.fn("geometry").remote(dict(model=HF[MAIN], spec=spec, runs=runs_geo, tag=f"geo_{MAIN}")); L = geo["layers"]; mid = L // 2
print("trait–trait cosine at mid layer:", geo["traits"]); print(np.array(geo["trait_cos"][mid]).round(2))
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
for tag, v in geo["runs"].items():
    if not tag.endswith(f"_{FINAL}"): continue
    for a, t in zip(ax, geo["traits"]): a.plot(v["proj"][t], label=tag)
for a, t in zip(ax, geo["traits"]): a.set_title(f"RL shift on {t} (1 = full persona contrast)", fontsize=8); a.set_xlabel("layer")
ax[0].legend(fontsize=6); plt.show()

In [ ]:
pts = []
b0 = float(fin[fin.run == "base"].abstain_recall.iloc[0])
for tag, v in geo["runs"].items():
    if not tag.endswith(f"_{FINAL}"): continue
    e = fin[fin.run == tag[:-len(str(FINAL)) - 1]]
    if len(e): pts.append((v["proj"]["humility"][mid], float(e.abstain_recall.iloc[0]) - b0, v["proj"]["effort"][mid]))
Pt = np.array(pts); print("proj_humility, Δabstain, proj_effort\n", Pt.round(3))
plt.scatter(Pt[:, 0], Pt[:, 1]); plt.xlabel("projection of RL shift on humility vector"); plt.ylabel("Δ abstain recall vs base"); plt.show()

### Steering × RL dissociation (the central causal test)
Terseness reproduces the drift ⇒ the entanglement lives in the effort axis. Adding effort back restores abstention ⇒ the effort axis **mediates** it. Neither ⇒ RL changed other things.

In [ ]:
hi = P["stage1"][-1]; r1 = f"{hi[0]}_lam{hi[1]}_s{hi[2]}"
for al in P["steer_base"]:
    t = f"st_base_{al}"; local.launch("eval_ckpt", dict(name="base", ckpt=None, model=HF[MAIN], tier=P["tier_final"], spec=spec, steer=dict(trait="effort", alpha=al), out_tag=t), key=t)
for al in P["steer_rl"]:
    t = f"st_{r1}_{al}"; local.launch("eval_ckpt", dict(name=r1, ckpt=FINAL, model=HF[MAIN], tier=P["tier_final"], spec=spec, steer=dict(trait="effort", alpha=al), out_tag=t), key=t)
print(local.wait("st_", every=90)); local.pull("/eval")
S = {os.path.basename(f)[3:-5]: json.load(open(f))["metrics"] for f in local.find("st_*.json")}
S["base (no steer)"] = json.load(open(local.find(f"ev_{MAIN}_base.json")[0]))["metrics"]; S[f"{r1} (no steer)"] = json.load(open(local.find(f"ev_{r1}_{FINAL}.json")[0]))["metrics"]
T = pd.DataFrame(S).T; T[[c for c in ["acc_id", "len_id", "abstain_recall", "fabricate", "flip_user", "flip_peer", "calib_auroc"] if c in T]].round(3)